# Train FPL Points Prediction Model
This notebook focuses on training a model to predict FPL points using the features from the previous notebooks.

## 1. Include required libraries

In [2]:
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import joblib

## 2. Read Data
We read the CSV files generated by earlier steps into Spark dataframes:
* features/XXX.csv

In [8]:
# Load the dataset
input_filepath = "../data/processed/features/20250329_004959/features.parquet"
data = pd.read_parquet(input_filepath)

# Show data
data.head()

,recent_goals_scored,long_term_goals_scored,recent_goals_conceded,long_term_goals_conceded,recent_assists,long_term_assists,recent_expected_assists,long_term_expected_assists,recent_expected_goal_involvements,long_term_expected_goal_involvements,...,recent_team_total_points,long_term_team_total_points,position_GKP,position_DEF,position_MID,position_FWD,position_GK,was_home_true,was_home_false,total_points
0,0,0,1,0,0,0,0.15,0.0,0.31,0.0,...,199,492,0,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,166,203,1,0,0,0,1,1,0,0
2,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,236,840,1,0,0,0,1,1,0,0
3,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,236,840,1,0,0,0,1,1,0,0
4,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,278,728,1,0,0,0,1,1,0,0


## 3. Data Cleaning
We'll handle data cleaning tasks such as dealing with missing values.

In [9]:
# Drop rows with missing values
data = data.dropna()

# Verify no missing values
print("Missing values after handling:\n", data.isnull().sum())

# Show the data
data.head()

Missing values after handling:
 recent_goals_scored         0
long_term_goals_scored      0
recent_goals_conceded       0
long_term_goals_conceded    0
recent_assists              0
                           ..
position_FWD                0
position_GK                 0
was_home_true               0
was_home_false              0
total_points                0
Length: 84, dtype: int64


,recent_goals_scored,long_term_goals_scored,recent_goals_conceded,long_term_goals_conceded,recent_assists,long_term_assists,recent_expected_assists,long_term_expected_assists,recent_expected_goal_involvements,long_term_expected_goal_involvements,...,recent_team_total_points,long_term_team_total_points,position_GKP,position_DEF,position_MID,position_FWD,position_GK,was_home_true,was_home_false,total_points
0,0,0,1,0,0,0,0.15,0.0,0.31,0.0,...,199,492,0,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,166,203,1,0,0,0,1,1,0,0
2,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,236,840,1,0,0,0,1,1,0,0
3,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,236,840,1,0,0,0,1,1,0,0
4,0,0,0,0,0,0,0.00,0.0,0.00,0.0,...,278,728,1,0,0,0,1,1,0,0


## 4. Train Model
We'll train the model.

In [11]:
# Prepare the data for the model
X = data.drop('total_points', axis=1)
y = data['total_points']

# Define cross-validation parameters
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize models
linear_model = LinearRegression()

# Perform cross-validation for Linear Regression
linear_mse_scores = -cross_val_score(linear_model, X, y, cv=cv, scoring='neg_mean_squared_error')
linear_r2_scores = cross_val_score(linear_model, X, y, cv=cv, scoring='r2')

# Print the cross-validation results
print("Linear Regression Cross-Validation Results:")
print(f"Average MSE: {np.mean(linear_mse_scores)}")
print(f"Average R-squared: {np.mean(linear_r2_scores)}")

Linear Regression Cross-Validation Results:
Average MSE: 3.772779133610933
Average R-squared: 0.32835856646668293


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Prepare the data for the model
X = data.drop('total_points', axis=1)
y = data['total_points']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Regressor model
rf_model = RandomForestRegressor(random_state=42, n_estimators=10)

# Train the model on the training data
rf_model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_rf = rf_model.predict(X_val)

# Calculate the Mean Squared Error (MSE)
rf_mse = mean_squared_error(y_val, y_pred_rf)

# Calculate the R-squared (R^2) score
rf_r2 = r2_score(y_val, y_pred_rf)

# Print the validation results
print("\nRandom Forest Regression Validation Results:")
print(f"Mean Squared Error: {rf_mse}")
print(f"R-squared: {rf_r2}")


Random Forest Regression Validation Results:
Mean Squared Error: 3.6624669060983592
R-squared: 0.32882305259130207


In [ ]:
# Prepare the data for the model
X = data.drop('total_points', axis=1)
y = data['total_points']

# Define cross-validation parameters
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize models
rf_model = RandomForestRegressor(random_state=42)

# Perform cross-validation for Random Forest Regression
rf_mse_scores = -cross_val_score(rf_model, X, y, cv=cv, scoring='neg_mean_squared_error')
rf_r2_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='r2')

print("\nRandom Forest Regression Cross-Validation Results:")
print(f"Average MSE: {np.mean(rf_mse_scores)}")
print(f"Average R-squared: {np.mean(rf_r2_scores)}")

KeyboardInterrupt: 

In [ ]:
# Retrain the best model on the entire dataset
# For this example, let's assume Random Forest performed better
best_model = RandomForestRegressor(random_state=42)
best_model.fit(X, y)

## 6. Export Model
Finally, we'll export the trained model.

In [16]:
from datetime import datetime
import os

# Define output directory
# We define the directory where the processed data will be saved.
data_source = "random_forest"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/model/{data_source}/{current_datetime}"
filename = 'model.joblib'
filepath = os.path.join(output_dir, filename)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the model
joblib.dump(rf_model, filepath)

print(f"Model training complete. Model saved to {output_dir}")

Model training complete. Model saved to ../data/model/random_forest/20250329_011922
